___
# <center>Regressão logística</center>
___

## Aula 20

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * explicar por que a regressão linear não serve para uma variável resposta de
   sim ou não;
 * passar de probabilidade para chance e calcular uma razão de chances numa
   tabela 2x2;
 * ajustar uma regressão logística no statsmodels e interpretar cada coeficiente
   como razão de chances;
 * conduzir uma análise completa, da pergunta à limitação, e relatá-la em
   poucas frases.

___
<div id="indice"></div>

## Índice

- [Por que não a regressão linear](#linear)
- [Chance e razão de chances](#chance)
- [Regressão logística](#logistica)
- [Probabilidade prevista](#previsao)
- [Análise completa](#completa)

> **Sobre a base.** Os dados são do projeto da clínica com o BTG e estão cobertos
> por sigilo bancário e pelo termo de parceria. O arquivo `btg_aulas.csv` está no
> BlackBoard e **não pode ser compartilhado** fora da turma, nem subido para o
> GitHub, nem colado em ferramentas externas além das usadas em aula.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Baixe btg_aulas.csv do BlackBoard e deixe na mesma pasta deste notebook.
# No Colab: clique no ícone de pasta, à esquerda, e arraste o arquivo para lá.
processos = pd.read_csv("btg_aulas.csv")
processos.shape

[Volta ao Índice](#indice)

___
<div id="linear"></div>

# Por que não a regressão linear

`pagou` é sim ou não. Se virar 1 e 0, a regressão linear roda, mas a reta não
sabe que a resposta fica entre 0 e 1.

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

base = processos.dropna(subset=["valor_financiado"]).copy()
base["y"] = base["pagou"].astype(int)
base["financiado_mil"] = base["valor_financiado"] / 1000
base["recente"] = base["ano_ajuizamento"] >= 2022

linear = smf.ols("y ~ financiado_mil", data=base).fit()
linear.predict(pd.DataFrame({"financiado_mil": [10, 50, 100, 150]})).round(3)

Para um contrato de R$ 100 mil a reta prevê uma probabilidade **negativa**. A
regressão logística resolve isso modelando a **chance** na escala do logaritmo,
e a curva fica sempre entre 0 e 1.

[Volta ao Índice](#indice)

___
<div id="chance"></div>

# Chance e razão de chances

$$\text{chance} = \frac{p}{1 - p}$$

20% de probabilidade é chance de 0,25: 1 processo que paga para cada 4 que não
pagam.

In [ ]:
pd.crosstab(base["recente"], base["pagou"], margins=True)

In [ ]:
antigos = base[~base["recente"]]
recentes = base[base["recente"]]

chance_antigos = antigos["pagou"].sum() / (~antigos["pagou"]).sum()
chance_recentes = ____ / ____
razao = chance_recentes / chance_antigos
print(f"chance: antigos {chance_antigos:.3f}, recentes {chance_recentes:.3f}, razão {razao:.3f}")

Nos processos de 2022 em diante, a chance de pagamento é **0,31 vez** a dos
antigos, ou 69% menor. A probabilidade cai de 21,5% para 7,9%: 63% menor. Razão
de chances e razão de probabilidades só são parecidas quando as
probabilidades são pequenas.

[Volta ao Índice](#indice)

___
<div id="logistica"></div>

# Regressão logística

$$\log\left(\frac{p}{1-p}\right) = a + b_1 x_1 + b_2 x_2 + \dots$$

O coeficiente $b$ está na escala do log da chance. **$e^b$ é a razão de
chances**: quanto a chance é multiplicada quando $x$ aumenta uma unidade,
mantidas as demais constantes.

In [ ]:
m1 = smf.logit("y ~ recente", data=base).fit(disp=0)
print(m1.summary().tables[1])
print("razão de chances:", np.exp(m1.params).round(3).to_dict())

A mesma razão de 0,31 da tabela. Agora com mais variáveis:

In [ ]:
m2 = smf.logit("y ~ recente + financiado_mil + ____", data=base).fit(disp=0)

In [ ]:
def razoes(modelo):
    ic = np.exp(modelo.conf_int())
    return pd.DataFrame({"razao_chances": np.exp(modelo.params), "ic_baixo": ic[0],
                         "ic_alto": ic[1], "valor_p": modelo.pvalues}).round(3)

razoes(m2)

**Como relatar:**

* `recente`: mantidos o valor financiado e a contestação, a chance de pagamento
  nos processos de 2022 em diante é 0,37 vez a dos antigos.
* `financiado_mil`: cada R$ 1.000 a mais financiados multiplicam a chance por
  0,975, ou seja, 2,5% a menos. Para R$ 10 mil a mais: $0{,}975^{10} \approx 0{,}78$.
* `tem_contestacao`: razão de 0,99, com intervalo de 0,58 a 1,71. Sozinha, a
  contestação parecia reduzir o pagamento; **mantida a idade do processo, a
  diferença some**.

[Volta ao Índice](#indice)

___
<div id="previsao"></div>

# Probabilidade prevista

In [ ]:
perfis = pd.DataFrame({
    "recente": [False, True, False, True],
    "financiado_mil": [20, 20, 50, 50],
    "tem_contestacao": [True, True, True, True],
})
perfis["prob_prevista"] = m2.predict(perfis).round(3)
perfis

A probabilidade prevista é o que vira decisão: na aula 21 vamos escolher um
**ponto de corte** e contar os erros de cada lado.

[Volta ao Índice](#indice)

___
<div id="completa"></div>

# Análise completa

Pergunta: **processos de contratos já liquidados terminam menos em pagamento do
que os de contratos em atraso?**

Siga as etapas. A coleta e a estruturação já foram feitas nas aulas 16 e 17 (os
PDFs viraram colunas); aqui você faz da exploratória à limitação.

**1. Exploratória.** A taxa de pagamento por situação do contrato, com o
tamanho de cada grupo.

In [ ]:
base.groupby("____")["pagou"].agg(["size", "mean"]).round(3)

**2. Modelo.** Regressão logística com a situação do contrato, ajustada pela
idade do processo e pelo valor financiado. A referência é `ATRASO`.

In [ ]:
m3 = smf.logit("y ~ C(status_contrato) + ____ + ____", data=base).fit(disp=0)
razoes(m3)

**3. Previsão.** A probabilidade prevista para um contrato liquidado e um em
atraso, ambos de R$ 25 mil, em ação de 2023.

In [ ]:
# item 3

**4. Relatório**, em três frases, na célula abaixo: a pergunta, o resultado com
a razão de chances e o intervalo, e **uma** limitação que um advogado do banco
apontaria.

*(escreva aqui)*

[Volta ao Índice](#indice)